# Assignment 3: Informed Search

##  Grid Navigation
In this section, you will investigate the problem of navigation on a two dimensional grid with obstacles. The goal is to produce the shortest path between a provided pair of points, taking care to maneuver around the obstacles as needed. Path length is measured in Euclidean distance. Valid directions of movement include up, down, left, right, up-left, up-right, down-left, and down-right. 

### Note:
You are expected to write code where you see **your code here**.  
Make sure you delete the lines with **pass** and **raise NotImplementedError** or your code may not run correctly.

### Task
Your task is to write a function find_path(start, goal, scene) which returns the shortest path from the start point to the goal point that avoids traveling through the obstacles in the grid. 

### Structure/Representation
For this problem, points will be represented as two-element tuples of the form (row, column), and scenes will be represented as two-dimensional lists of Boolean values, with False values corresponding empty spaces and True values corresponding to obstacles. 

### Output
Your output should be the list of points in the path, and should explicitly include both the start point and the goal point. If multiple optimal solutions exist, any of them may be returned. If no optimal solutions exist, or if the start point or goal point lies on an obstacle, you should return the sentinal value None. If start and goal state are the same, return None.

### Implementation
Your implementation should consist of 
* an A* search 
* the straight-line Euclidean distance heuristic. 

Helper functions are allowed and encouraged.

## 1. Euclidean distance
First, let's define a function used for euclidean distance.

In [1]:
############################################################
# Section 1 Grid Navigation - Euclidiean distance
############################################################
def grid_euclidean_distance(current_state, goal_state):
    """Calculate Euclidean distance between two points."""
    return ((current_state[0] - goal_state[0])**2 + (current_state[1] - goal_state[1])**2)**0.5


In [2]:
##########################
### TEST YOUR SOLUTION ###
##########################

current_state1, goal_state1 = (1,1), (1,1)
assert grid_euclidean_distance(current_state1, goal_state1) == 0

current_state2, goal_state2 = (1,2), (1,1)
assert grid_euclidean_distance(current_state2, goal_state2) == 1

current_state3, goal_state3 = (3,2), (1,2)
assert grid_euclidean_distance(current_state3, goal_state3) == 2

current_state4, goal_state4 = (1,1), (2,2)
assert grid_euclidean_distance(current_state4, goal_state4) == 2**0.5
print("test passed!")

test passed!


## 2. Helper functions
Next, let's define a functions that finds the successors.

In [3]:
def grid_successors(current, scene):
    """
    Find all valid successor states from current position.
    Valid moves: up, down, left, right, and all 4 diagonals.
    Returns tuple of valid positions (not on obstacles and within bounds).
    """
    rows = len(scene)
    cols = len(scene[0]) if rows > 0 else 0
    row, col = current
    
    # All 8 directions: up, down, left, right, up-left, up-right, down-left, down-right
    directions = [
        (-1, 0),   # up
        (1, 0),    # down
        (0, -1),   # left
        (0, 1),    # right
        (-1, -1),  # up-left
        (-1, 1),   # up-right
        (1, -1),   # down-left
        (1, 1)     # down-right
    ]
    
    successors = []
    for dr, dc in directions:
        new_row, new_col = row + dr, col + dc
        # Check if within bounds and not an obstacle
        if 0 <= new_row < rows and 0 <= new_col < cols and not scene[new_row][new_col]:
            successors.append((new_row, new_col))
    
    return tuple(successors)


In [4]:
scene1 = [[True, True, True],
         [False, False, True]]
assert grid_successors((1, 2), scene1) == ((1, 1),)
assert grid_successors((0, 1), scene1) == ((1, 1),(1, 0),)
print("test passed!")

test passed!


## 3. Find path
Finally let's implement the path search.

In [5]:
############################################################
# Grid Navigation
############################################################
import collections, itertools, queue, random, copy

 
def find_path(start, goal, scene):
    """
    A* search to find shortest path from start to goal avoiding obstacles.
    Returns list of points in path, or None if no path exists.
    """
    # Check if start or goal is on an obstacle
    if scene[start[0]][start[1]] or scene[goal[0]][goal[1]]:
        return None
    
    # Check if start and goal are the same
    if start == goal:
        return None
    
    # Priority queue: (f_score, counter, current_pos, path)
    # counter is used to break ties in priority queue
    pq = queue.PriorityQueue()
    counter = itertools.count()
    
    # g_score: cost from start to current
    # f_score: g_score + heuristic (estimated total cost)
    initial_h = grid_euclidean_distance(start, goal)
    pq.put((initial_h, next(counter), start, [start], 0))
    
    # Keep track of visited nodes with their best g_score
    visited = {start: 0}
    
    while not pq.empty():
        f_score, _, current, path, g_score = pq.get()
        
        # If we reached the goal, return the path
        if current == goal:
            return path
        
        # Skip if we've already found a better path to this node
        if current in visited and visited[current] < g_score:
            continue
        
        # Explore successors
        for successor in grid_successors(current, scene):
            # Calculate new g_score (actual cost from start to successor)
            new_g_score = g_score + grid_euclidean_distance(current, successor)
            
            # Only consider this path if it's better than any previous path to successor
            if successor not in visited or new_g_score < visited[successor]:
                visited[successor] = new_g_score
                h_score = grid_euclidean_distance(successor, goal)
                new_f_score = new_g_score + h_score
                new_path = path + [successor]
                pq.put((new_f_score, next(counter), successor, new_path, new_g_score))
    
    # No path found
    return None


In [6]:
##########################
### TEST YOUR SOLUTION ###
##########################
scene1 = [[False, False, False], 
          [False, True , False], 
          [False, False, False]] 

assert find_path((0, 0), (2, 1), scene1) == [(0, 0), (1, 0), (2, 1)] 

scene2 = [[False, True, False], 
          [False, True, False], 
          [False, True, False]] 
assert find_path((0, 0), (0, 2), scene2) is None
print("test passed!")

test passed!
